# 200-Day SMA Regime Filter (Bull/Bear) on SPY
## Strategy Brief -- The strategy uses the 200-day Simple Moving Average (SMA) to determine the market regime. When the SPY price is above the 200-day SMA, it indicates a bull market, and the strategy takes a long position. Conversely, when the SPY price is below the 200-day SMA, it indicates a bear market, and the strategy takes a short position. This simple yet effective approach aims to capitalize on long-term trends while minimizing exposure during downturns.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
In this phase, we define the parameters and constants that will be used throughout the strategy implementation.

In [ ]:
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
TICKER = 'SPY'
SMA_PERIOD = 200

### PHASE 2 - Data Exploration
We will download historical price data for SPY and compute the 200-day SMA. The SMA will be plotted along with the price to visualize the market regime.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(TICKER, start=START_DATE, end=END_DATE)

# Compute 200-day SMA
data['SMA_200'] = data['Close'].rolling(window=SMA_PERIOD).mean()

# Plot price and SMA
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close', color='black')
plt.plot(data['SMA_200'], label='200-Day SMA', color='blue')
plt.title('SPY Price and 200-Day SMA')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.show()

### PHASE 3 - Strategy Engineering
We will create a signal series based on the position of the price relative to the 200-day SMA. The signal will indicate whether to be long or short.

In [ ]:
# Create signal: 1 for long, -1 for short
data['Signal'] = np.where(data['Close'] > data['SMA_200'], 1, -1)

# Entry/Exit logic
# Shift the signal to avoid lookahead bias
data['Position'] = data['Signal'].shift(1)

### PHASE 4 - Coding & Backtesting
We will calculate daily returns based on the positions and plot the equity curve to visualize the performance of the strategy.

In [ ]:
# Calculate daily returns
data['Market_Returns'] = data['Close'].pct_change()
data['Strategy_Returns'] = data['Market_Returns'] * data['Position']

# Calculate equity curve
data['Equity_Curve'] = (1 + data['Strategy_Returns']).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Equity_Curve'], label='Strategy Equity Curve', color='green')
plt.title('Strategy Equity Curve')
plt.xlabel('Date')
plt.ylabel('Equity')
plt.legend()
plt.show()

### PHASE 5 - Performance Evaluation
We will calculate key performance metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. We will also compare the strategy's performance against a buy-and-hold approach.

In [ ]:
def calculate_performance_metrics(data):
    # Calculate CAGR
    start_value = 1
    end_value = data['Equity_Curve'].iloc[-1]
    n_years = (data.index[-1] - data.index[0]).days / 365.25
    cagr = (end_value / start_value) ** (1 / n_years) - 1

    # Calculate Sharpe ratio
    sharpe_ratio = data['Strategy_Returns'].mean() / data['Strategy_Returns'].std() * np.sqrt(252)

    # Calculate Sortino ratio
    downside_std = data[data['Strategy_Returns'] < 0]['Strategy_Returns'].std()
    sortino_ratio = data['Strategy_Returns'].mean() / downside_std * np.sqrt(252)

    # Calculate Calmar ratio
    max_drawdown = (data['Equity_Curve'].cummax() - data['Equity_Curve']).max()
    calmar_ratio = cagr / max_drawdown

    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

# Performance metrics
cagr, sharpe, sortino, calmar, max_drawdown = calculate_performance_metrics(data)

# Buy-and-hold comparison
buy_and_hold_cagr = (data['Close'].iloc[-1] / data['Close'].iloc[0]) ** (1 / n_years) - 1

# Print results
print(f"CAGR: {cagr:.2%}")
print(f"Sharpe Ratio: {sharpe:.2f}")
print(f"Sortino Ratio: {sortino:.2f}")
print(f"Calmar Ratio: {calmar:.2f}")
print(f"Max Drawdown: {max_drawdown:.2%}")
print(f"Buy-and-Hold CAGR: {buy_and_hold_cagr:.2%}")

### PHASE 6 - Deploy & Monitor
We will create a function to download the last 60 days of data and compute today's signal to determine the current position.

In [ ]:
def get_current_position(ticker, sma_period):
    # Download last 60 days of data
data = yf.download(ticker, period='60d')
    
    # Compute 200-day SMA
data['SMA_200'] = data['Close'].rolling(window=sma_period).mean()
    
    # Determine current signal
    if data['Close'].iloc[-1] > data['SMA_200'].iloc[-1]:
        position = 'Long'
    else:
        position = 'Short'
    
    print(f"Current position for {ticker}: {position}")

# Check current position
get_current_position(TICKER, SMA_PERIOD)